# ⚡ SkinCheck AI - Multi-Task Skin Condition Trainer (Kaggle Edition)
Model ini dilatih untuk mengidentifikasi 4 kondisi kulit: **Jerawat, Kemerahan, Hyperpigmentasi, dan Tekstur Kulit**.


In [ ]:
# 1. Setup Environment & Direktori Kerja
import os
import sys
import json
import time
import shutil
import hashlib
from pathlib import Path
import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split

print("CUDA Available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Perangkat komputasi aktif: {device}")

BASE_DIR = Path("/kaggle/working/skincheck_data")
MASTER_DIR = BASE_DIR / "master_dataset"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
EXPORT_DIR = Path("/kaggle/working/exports")

CATEGORIES = ["acne", "redness", "hyperpigmentation", "texture", "normal_skin"]
for cat in CATEGORIES:
    (MASTER_DIR / cat).mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print("📁 Direktori kerja siap!")


In [ ]:
# 2. Pemindaian & Penimbunan Dataset dari /kaggle/input/
print("🔍 Memindai gambar dari dataset di /kaggle/input/ ...")
valid_exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
input_images = []

input_dir = Path("/kaggle/input")
if input_dir.exists():
    input_images = [p for p in input_dir.rglob("*.*") if p.suffix.lower() in valid_exts]

print(f"📊 Ditemukan {len(input_images)} file gambar di /kaggle/input/")

CATEGORY_KEYWORDS = {
    "acne": ["acne", "jerawat", "comedo", "pimple", "papule", "pustule", "nodule", "whitehead", "blackhead"],
    "redness": ["redness", "erythema", "rosacea", "merah", "kemerahan", "rash", "eczema", "blush", "dermatitis"],
    "hyperpigmentation": ["hyperpigmentation", "pigment", "melasma", "dark_spot", "flek", "sunspot", "freckles", "nevus", "melanoma"],
    "texture": ["texture", "wrinkle", "pore", "pori", "rough", "kasar", "keriput", "sagging", "keratosis", "atopic"],
    "normal_skin": ["normal", "clear", "healthy", "clean", "sehat", "mulus"]
}

cat_counts = {c: 0 for c in CATEGORIES}

def detect_cat(path_obj):
    p_str = str(path_obj).lower().replace("\\", "/")
    for cat, kws in CATEGORY_KEYWORDS.items():
        for kw in kws:
            if kw in p_str:
                return cat
    # Distribusi seimbang jika tidak cocok keyword spesifik
    return min(cat_counts, key=cat_counts.get)

# Salin dan normalisasi gambar (maksimal 250 per kategori agar cepat dan seimbang)
max_per_cat = 250
processed = 0

for p in input_images:
    cat = detect_cat(p)
    if cat_counts[cat] >= max_per_cat:
        continue
    try:
        cv_img = cv2.imread(str(p))
        if cv_img is None or cv_img.shape[0] < 64 or cv_img.shape[1] < 64:
            continue
        target_path = MASTER_DIR / cat / f"{cat}_{cat_counts[cat]:04d}.jpg"
        cv2.imwrite(str(target_path), cv_img, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
        cat_counts[cat] += 1
        processed += 1
    except Exception:
        pass

# Jika dataset eksternal sedikit, buat variasi augmentasi dasar agar sampel minimal tercapai
for cat in CATEGORIES:
    cat_files = list((MASTER_DIR / cat).glob("*.jpg"))
    if len(cat_files) < 30 and len(cat_files) > 0:
        base_img = cv2.imread(str(cat_files[0]))
        for i in range(len(cat_files), 40):
            aug = cv2.flip(base_img, 1) if i % 2 == 0 else cv2.GaussianBlur(base_img, (5, 5), 0)
            cv2.imwrite(str(MASTER_DIR / cat / f"{cat}_aug_{i:04d}.jpg"), aug)
            cat_counts[cat] += 1

print("✅ Penimbunan data selesai! Jumlah sampel per kondisi:")
for cat in CATEGORIES:
    n = len(list((MASTER_DIR / cat).glob("*.jpg")))
    print(f" - {cat}: {n} foto")


In [ ]:
# 3. Multi-Angle & Distance Augmentation Pipeline
class MultiAngleTransform:
    def __init__(self, is_train=True):
        if is_train:
            self.t = T.Compose([
                T.RandomResizedCrop(224, scale=(0.5, 1.15)),
                T.RandomHorizontalFlip(p=0.5),
                T.RandomRotation(degrees=(-25, 25)),
                T.RandomPerspective(distortion_scale=0.3, p=0.6),
                T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2),
                T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])
        else:
            self.t = T.Compose([
                T.Resize((240, 240)),
                T.CenterCrop(224),
                T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])
    def __call__(self, img):
        return self.t(img)

class MultiTaskSkinDataset(Dataset):
    def __init__(self, master_dir, transform=None):
        self.samples = []
        self.transform = transform
        for cat in CATEGORIES:
            cdir = master_dir / cat
            for f in cdir.glob("*.jpg"):
                targets = {
                    "acne": 0.85 if cat == "acne" else 0.08,
                    "redness": 0.85 if cat == "redness" else (0.35 if cat == "acne" else 0.08),
                    "hyperpigmentation": 0.85 if cat == "hyperpigmentation" else 0.08,
                    "texture": 0.85 if cat == "texture" else (0.45 if cat == "acne" else 0.10)
                }
                if cat == "normal_skin":
                    targets = {"acne": 0.05, "redness": 0.05, "hyperpigmentation": 0.05, "texture": 0.05}
                self.samples.append((f, targets))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, tgts = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        t_target = torch.tensor([tgts["acne"], tgts["redness"], tgts["hyperpigmentation"], tgts["texture"]], dtype=torch.float32)
        return img, t_target

dataset = MultiTaskSkinDataset(MASTER_DIR, transform=MultiAngleTransform(is_train=True))
print(f"📊 Total dataset siap latih: {len(dataset)} gambar")


In [ ]:
# 4. Arsitektur Multi-Task Neural Network (MobileNetV3)
class SkinCheckMultiTaskNet(nn.Module):
    def __init__(self):
        super(SkinCheckMultiTaskNet, self).__init__()
        base = models.mobilenet_v3_large(weights=None) # Offline resilient, no download needed!
        self.features = base.features
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        f_dim = 960
        
        self.acne_head = self._make_head(f_dim)
        self.redness_head = self._make_head(f_dim)
        self.hyperpigmentation_head = self._make_head(f_dim)
        self.texture_head = self._make_head(f_dim)
        
    def _make_head(self, in_dim):
        return nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(),
            nn.Dropout(p=0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        feat = self.features(x)
        feat = self.avgpool(feat)
        feat = torch.flatten(feat, 1)
        
        acne = self.acne_head(feat) * 100.0
        redness = self.redness_head(feat) * 100.0
        pigment = self.hyperpigmentation_head(feat) * 100.0
        texture = self.texture_head(feat) * 100.0
        return torch.cat([acne, redness, pigment, texture], dim=1)

model = SkinCheckMultiTaskNet().to(device)
print(f"🚀 Model MultiTaskNet terinisialisasi pada {device}!")


In [ ]:
# 5. Pelatihan Model (5 Epochs)
BATCH_SIZE = 16
EPOCHS = 5

val_size = max(1, int(0.2 * len(dataset)))
train_size = len(dataset) - val_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

best_val_loss = float("inf")
best_model_path = CHECKPOINT_DIR / "best_model.pth"

print(f"🚀 Memulai pelatihan {EPOCHS} Epochs...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for imgs, tgts in train_loader:
        imgs = imgs.to(device)
        tgts = (tgts * 100.0).to(device)
        
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, tgts)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        
    train_loss = running_loss / len(train_set)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, tgts in val_loader:
            imgs = imgs.to(device)
            tgts = (tgts * 100.0).to(device)
            preds = model(imgs)
            val_loss += criterion(preds, tgts).item() * imgs.size(0)
    val_loss = val_loss / len(val_set)
    
    print(f"Epoch [{epoch}/{EPOCHS}] -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"  ⭐ Best model disimpan! Val Loss: {best_val_loss:.4f}")

print("🎉 Pelatihan selesai!")


In [ ]:
# 6. Ekspor Model ke ONNX & ZIP untuk Antigravity
print("📦 Mengekspor model untuk Antigravity...")
model.eval()
model.to("cpu")

dummy_input = torch.randn(1, 3, 224, 224, dtype=torch.float32)

# 1. Simpan ke EXPORT_DIR dan langsung ke /kaggle/working/
onnx_export_path = Path("/kaggle/working/skincheck_model.onnx")
torch.onnx.export(
    model,
    dummy_input,
    str(onnx_export_path),
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=["input_image"],
    output_names=["skin_indicators"],
    dynamic_axes={"input_image": {0: "batch_size"}, "skin_indicators": {0: "batch_size"}}
)
shutil.copy(str(onnx_export_path), str(EXPORT_DIR / "skincheck_model.onnx"))
print("✅ skincheck_model.onnx berhasil dibuat!")

# 2. Export TorchScript .pt
pt_path = Path("/kaggle/working/skincheck_model.pt")
traced = torch.jit.trace(model, dummy_input)
traced.save(str(pt_path))
shutil.copy(str(pt_path), str(EXPORT_DIR / "skincheck_model.pt"))
print("✅ skincheck_model.pt berhasil dibuat!")

# 3. Metadata JSON
metadata = {
    "model_name": "SkinCheck-MultiTask-Net",
    "version": "1.0.0",
    "architecture": "MobileNetV3-Large MultiTask",
    "input_size": [224, 224, 3],
    "input_mean": [0.485, 0.456, 0.406],
    "input_std": [0.229, 0.224, 0.225],
    "indicators": [
        {"name": "acne", "label": "Jerawat", "scale": [0, 100]},
        {"name": "redness", "label": "Kemerahan", "scale": [0, 100]},
        {"name": "hyperpigmentation", "label": "Hyperpigmentasi", "scale": [0, 100]},
        {"name": "texture", "label": "Tekstur Kulit", "scale": [0, 100]}
    ],
    "non_medical_notice": "Estimated cosmetic visual indicators only, not for medical diagnosis."
}

meta_path = Path("/kaggle/working/model_metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)
shutil.copy(str(meta_path), str(EXPORT_DIR / "model_metadata.json"))
print("✅ model_metadata.json berhasil dibuat!")

# 4. ZIP Package
zip_base = "/kaggle/working/skincheck_ai_exported"
shutil.make_archive(zip_base, 'zip', EXPORT_DIR)
print(f"🎁 skincheck_ai_exported.zip berhasil dibuat di: {zip_base}.zip")
print("🚀 Seluruh berkas ekspor siap diunduh dari Kaggle Output!")
